# Qwen 后训练 (QLoRA + SFT)

一键在 Colab 或 Kaggle 上微调千问模型。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/greathousesh/post-training/blob/main/notebooks/train_qwen.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/post-training/blob/main/notebooks/train_qwen.ipynb)

**使用前**：把这个 notebook 里所有的 `greathousesh/post-training` 改成你自己的仓库地址（包括上面两个 badge 链接 和下方 git clone 那一格）。

**Kaggle 注意**：右侧 Settings → Accelerator 选 GPU (T4 x2)，Internet 打开（下载模型需要）。

**Colab 注意**：菜单栏 Runtime → Change runtime type → Hardware accelerator 选 GPU。

## 1. 环境检查

In [ ]:
!nvidia-smi

## 2. 安装依赖

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets pyyaml

## 3. 拉取项目代码

把 `REPO_URL` 改成你的 GitHub 仓库。

In [ ]:
import os

REPO_URL = "https://github.com/greathousesh/post-training.git"
REPO_DIR = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
os.chdir(REPO_DIR)
print("CWD:", os.getcwd())
!ls

## 4. 准备数据

数据格式：JSONL，每行一条对话，结构见 `data/example.jsonl`。

- **Colab**：下一格可以上传自己的 `train.jsonl`
- **Kaggle**：用右侧 `+ Add Input` 挂载一个 Dataset，然后把路径改成 `/kaggle/input/<dataset>/train.jsonl`
- **想先跑通流程**：跳过本步，直接用 `data/example.jsonl`

In [ ]:
# Colab 专用：上传自定义数据（取消注释使用）
# from google.colab import files
# import shutil
# uploaded = files.upload()
# for name in uploaded:
#     shutil.move(name, f"data/{name}")

!head -1 data/example.jsonl

## 5. 按 GPU 自适应调整 config

根据当前 GPU 显存自动选择模型大小和 batch 配置，改写 `config.yaml`。

In [ ]:
import yaml, torch

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {vram_gb:.1f} GB")

# 你也可以改下面这行指定训练数据
cfg["data"]["train_file"] = "data/example.jsonl"

if vram_gb < 20:            # T4 16GB / P100 16GB
    cfg["model"]["name"] = "Qwen/Qwen2.5-1.5B-Instruct"
    cfg["model"]["max_length"] = 1024
    cfg["training"]["per_device_train_batch_size"] = 1
    cfg["training"]["gradient_accumulation_steps"] = 16
elif vram_gb < 30:          # L4 22GB / V100 32GB
    cfg["model"]["name"] = "Qwen/Qwen2.5-7B-Instruct"
    cfg["model"]["max_length"] = 1024
    cfg["training"]["per_device_train_batch_size"] = 1
    cfg["training"]["gradient_accumulation_steps"] = 16
else:                       # A100 40/80GB
    cfg["model"]["name"] = "Qwen/Qwen2.5-7B-Instruct"

with open("config.yaml", "w") as f:
    yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)

print(yaml.dump(cfg, allow_unicode=True, sort_keys=False))

## 6. 训练

In [ ]:
!python train.py --config config.yaml

## 7. 导出 / 下载 LoRA adapter

In [ ]:
import os

OUTPUT_DIR = "outputs/qwen-sft-lora"

if not os.path.exists(OUTPUT_DIR):
    print(f"ERROR: {OUTPUT_DIR} 不存在 —— 上一格训练没跑完，先回去看训练那格的输出")
else:
    !ls {OUTPUT_DIR}
    !tar -czf qwen-lora.tar.gz {OUTPUT_DIR}

    # 平台检测用环境变量比 try/except 更稳
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        print("\nKaggle: /kaggle/working/qwen-lora.tar.gz 已就绪，右侧 Output 面板可下载")
    elif "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import files
        files.download("qwen-lora.tar.gz")
    else:
        print(f"\n本地: 压缩包在 {os.path.abspath('qwen-lora.tar.gz')}")

## 8. (可选) 合并 LoRA 到完整权重

合并需要以 bf16 加载完整 base model（7B 约 14GB），T4/P100 显存不够，建议在 A100 或本地做。

In [ ]:
# !python -m src.merge \
#     --base Qwen/Qwen2.5-7B-Instruct \
#     --adapter outputs/qwen-sft-lora \
#     --output outputs/qwen-sft-merged

## 9. 推送到 HuggingFace Hub

**上传前准备**：
1. 在 https://huggingface.co/settings/tokens 生成一个 **Write** 权限的 token
2. 运行下一格登录（把 token 粘到弹出的输入框）

**上传哪个**？
- **LoRA adapter**（几百 MB，推荐先传这个）：小、快，别人可以 `PeftModel.from_pretrained(base, 你的repo)` 加载
- **合并后的完整模型**（7B ≈ 14GB）：大，但可以在 HF 模型页直接用 Inference Widget 测试

Kaggle 免费盘 20GB，传 7B 合并模型会紧张；**先传 LoRA 最稳**。

In [ ]:
from huggingface_hub import login
login()  # 弹出输入框，粘 HF token

In [ ]:
# 方式 A：推 LoRA adapter（推荐先试这个）
HF_USERNAME = "your-hf-username"          # 改成你的 HF 用户名
REPO_NAME = "qwen-sft-lora-demo"           # 想要的仓库名

!python -m src.push_to_hf \
    --local outputs/qwen-sft-lora \
    --repo {HF_USERNAME}/{REPO_NAME}

# 方式 B：推合并后的完整模型（先跑过第 8 格，得到 outputs/qwen-sft-merged）
# !python -m src.push_to_hf \
#     --local outputs/qwen-sft-merged \
#     --repo {HF_USERNAME}/qwen-sft-merged-demo

### 在 HF 上测试

推完访问 `https://huggingface.co/<你的用户名>/<repo名>`：

- **合并模型**：模型页右侧自动有 Inference Widget，可以直接聊
- **LoRA adapter**：Widget 不直接支持，需要用代码加载：
  ```python
  from peft import PeftModel
  from transformers import AutoModelForCausalLM, AutoTokenizer

  base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
  tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
  model = PeftModel.from_pretrained(base, "<你的用户名>/<repo名>")
  ```
  或者建一个 HF Space 包成 Gradio 界面。